<p align="center">
• <a href="https://mayzune.com/"><strong>May Zune</strong></a> •
<a href="https://github.com/hellomayzune"><strong>GitHub</strong></a> •
<a href="https://orcid.org/0000-0003-0282-2633"><strong>ORCID</strong></a> •
<a href="https://scholar.google.com/citations?user=LmP8B_4AAAAJ&hl=en"><strong>Google Scholar</strong></a> •
<a href="https://www.researchgate.net/profile/May-Zune"><strong>ResearchGate</strong></a> •
<a href="https://www.linkedin.com/in/mayzune//"><strong>Linkedin</strong></a> •
</p>

# initial-data-OLS-regression

This notebook presents an **Ordinary Least Squares (OLS) regression analysis** applied to eight benchmark functions from a black-box optimisation study.  

The analysis investigates the relationships between the selected variables and evaluates the regression models using statistical significance and goodness-of-fit measures.

The notebook was developed during **Module 12: Bayesian Optimisation**.

## Summary of Findings

* The OLS results show that the **linear relationship between the input variables and the objective value varies considerably across the eight benchmark functions**.
* **Function 8** provides the strongest model fit, with **R² = 0.899** and adjusted R² = 0.873, indicating that the predictors explain approximately 90% of the observed variation.
* **Functions 4, 5, and 6** also show moderate-to-strong explanatory power, with R² values of **0.579, 0.662, and 0.647**, respectively.
* **Functions 1, 3, and 7** have comparatively weak overall model performance, with R² values of **0.037, 0.368, and 0.346**, respectively.
* **Function 2** is borderline significant overall (**p = 0.0525**), although **Input_1 is individually significant** (p = 0.035).
* Several predictors are statistically significant across the functions, indicating that their contribution to the objective value is stronger than that of other inputs.
* Diagnostic results suggest that the OLS assumptions are **better satisfied for some functions than others**; Functions 1 and 7 show notable departures from residual normality.
* Overall, the results indicate that **OLS can effectively capture linear relationships for some black-box functions, particularly Function 8, but its explanatory capability is highly function-dependent**.


### Comparative Dataset Overview

| **Function Sheet** | **Sample Size (N)** | **Num Inputs (d)** | **Output Mean** | **Output Std** | **Output Range [Min, Max]** | **Strongest Linear Driver** |
| ------------------ | ------------------: | -----------------: | --------------: | -------------: | --------------------------- | --------------------------- |
| **F1**             |                  10 |                  2 |       $-0.0004$ |       $0.0011$ | $[-0.0036, 0.0000]$         | `Input_2` ($r = -0.17$)     |
| **F2**             |                  10 |                  2 |        $0.2307$ |       $0.2376$ | $[-0.0656, 0.6112]$         | `Input_1` ($r = 0.75$)      |
| **F3**             |                  15 |                  3 |       $-0.1072$ |       $0.0872$ | $[-0.3989, -0.0348]$        | `Input_3` ($r = -0.57$)     |
| **F4**             |                  30 |                  4 |      $-17.2386$ |       $7.1380$ | $[-32.6257, -4.0255]$       | `Input_1` ($r = -0.54$)     |
| **F5**             |                  20 |                  4 |      $151.2719$ |     $251.9556$ | $[0.1129, 1088.8596]$       | `Input_4` ($r = 0.57$)      |
| **F6**             |                  20 |                  5 |       $-1.4954$ |       $0.4607$ | $[-2.5712, -0.7143]$        | `Input_5` ($r = -0.58$)     |
| **F7**             |                  30 |                  6 |        $0.2196$ |       $0.3073$ | $[0.0027, 1.3650]$          | `Input_5` ($r = -0.38$)     |
| **F8**             |                  40 |                  8 |        $7.8153$ |       $0.9590$ | $[5.5922, 9.5985]$          | `Input_1` ($r = -0.63$)     |


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm


# Information of OLS Regression

`Dependent Variable`: In this case, the dependent variable is "Output", which is what we aim to predict.

`R-Squared` is the most important measurement produced by this summary. R-squared (0.1) indicates that 1% of the variance in the  "Output" is explained by the model. 

Adjusted R-squared is important for analysing multiple dependent variables’ efficacy on the model. 

Linear regression has the quality that your model’s R-squared value will never go down with additional variables, only equal or higher. 

Therefore, your model could look more accurate with multiple variables, even if they are poorly contributing. 

The adjusted R-squared penalises the R-squared formula based on the number of variables; therefore, a lower adjusted score may be telling you some variables are not contributing to your model’s R-squared properly.

===

`F-Statistic`: The F-statistic in linear regression compares your produced linear model for your variables against a model that replaces your variables’ effects with 0 to find out if your group of variables are statistically significant. 

To interpret this number correctly, using a chosen alpha value and an F-table is necessary.

`Prob (F-Statistic)`: uses this number to tell you the accuracy of the null hypothesis, or whether it is accurate that your variables’ effect is 0.

===

`Omnibus` describes the normalcy of the distribution of our residuals using skew and kurtosis as measurements. 

A 0 would indicate perfect normalcy. 

`Prob(Omnibus)` is a statistical test measuring the probability that the residuals are normally distributed. 

A 1 would indicate a perfectly normal distribution.

`Skew` is a measurement of symmetry in our data, with 0 being perfect symmetry. 

`Kurtosis` measures the peakiness of our data, or its concentration around 0 in a normal curve. Higher kurtosis implies fewer outliers.

===

`Durbin-Watson` is a measurement of homoscedasticity, or an even distribution of errors throughout our data. 

Heteroscedasticity would imply an uneven distribution, for example, as the data point grows higher, the relative error grows higher. 

Ideal homoscedasticity will lie between 1 and 2. 

`Jarque-Bera (JB)` and `Prob(JB)` are alternate methods of measuring the same value as Omnibus and Prob(Omnibus) using skewness and kurtosis. 

We use these values to confirm each other. 

Condition number is a measurement of the sensitivity of our model as compared to the size of changes in the data it is analysing. 

Multicollinearity is strongly implied by a high condition number. 

Multicollinearity is a term to describe two or more independent variables that are strongly related to each other and falsely affect our predicted variable by redundancy.

In [2]:
# Read the data
df1 = pd.read_excel(open("initial_data/CapstoneFunction_from_Numpy.xlsx", "rb"), sheet_name="F1")
df2 = pd.read_excel(open("initial_data/CapstoneFunction_from_Numpy.xlsx", "rb"), sheet_name="F2")
df3 = pd.read_excel(open("initial_data/CapstoneFunction_from_Numpy.xlsx", "rb"), sheet_name="F3")
df4 = pd.read_excel(open("initial_data/CapstoneFunction_from_Numpy.xlsx", "rb"), sheet_name="F4")
df5 = pd.read_excel(open("initial_data/CapstoneFunction_from_Numpy.xlsx", "rb"), sheet_name="F5")
df6 = pd.read_excel(open("initial_data/CapstoneFunction_from_Numpy.xlsx", "rb"), sheet_name="F6")
df7 = pd.read_excel(open("initial_data/CapstoneFunction_from_Numpy.xlsx", "rb"), sheet_name="F7")
df8 = pd.read_excel(open("initial_data/CapstoneFunction_from_Numpy.xlsx", "rb"), sheet_name="F8")

In [3]:
df1.head(2)

,Input_1,Input_2,Output
0,0.319404,0.762959,1.322677e-79
1,0.574329,0.879898,1.033078e-46


<a id = "1"></a><br>
# Function 1: Radiation field (2D array, 10 samples)

In [4]:
# Define independent variables (X) and dependent variable (y)
X1 = df1.drop(columns=['Output'])
y1 = df1['Output']

In [5]:
# Add a constant term to the independent variables
X1 = sm.add_constant(X1)
model1 = sm.OLS(y1, X1).fit() # Fit the OLS regression model
print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:                 Output   R-squared:                       0.037
Model:                            OLS   Adj. R-squared:                 -0.239
Method:                 Least Squares   F-statistic:                    0.1327
Date:                Sun, 09 Aug 2026   Prob (F-statistic):              0.878
Time:                        18:19:30   Log-Likelihood:                 54.288
No. Observations:                  10   AIC:                            -102.6
Df Residuals:                       7   BIC:                            -101.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0002      0.001      0.142      0.8

<a id = "2"></a><br>
# Function 2:  Unspecified field (2D array, 10 samples)

In [6]:
# Define independent variables (X) and dependent variable (y)
X2 = df2.drop(columns=['Output'])
y2 = df2['Output']

# Add a constant term to the independent variables
X2 = sm.add_constant(X2)
model2 = sm.OLS(y2, X2).fit() # Fit the OLS regression model
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                 Output   R-squared:                       0.569
Model:                            OLS   Adj. R-squared:                  0.446
Method:                 Least Squares   F-statistic:                     4.623
Date:                Sun, 09 Aug 2026   Prob (F-statistic):             0.0525
Time:                        18:19:30   Log-Likelihood:                 4.9206
No. Observations:                  10   AIC:                            -3.841
Df Residuals:                       7   BIC:                            -2.933
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1713      0.146     -1.176      0.2

<a id = "3"></a><br>
# Function 3:  A drug discovery project (3D array, 15 samples)

In [7]:
# Define independent variables (X) and dependent variable (y)
X3 = df3.drop(columns=['Output'])
y3 = df3['Output']

# Add a constant term to the independent variables
X3 = sm.add_constant(X3)
model3 = sm.OLS(y3, X3).fit() # Fit the OLS regression model
print(model3.summary())

                            OLS Regression Results                            
Dep. Variable:                 Output   R-squared:                       0.368
Model:                            OLS   Adj. R-squared:                  0.196
Method:                 Least Squares   F-statistic:                     2.136
Date:                Sun, 09 Aug 2026   Prob (F-statistic):              0.154
Time:                        18:19:30   Log-Likelihood:                 19.275
No. Observations:                  15   AIC:                            -30.55
Df Residuals:                      11   BIC:                            -27.72
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0439      0.070     -0.627      0.5

<a id = "4"></a><br>
# Function 4:  Warehouse business (4D array, 30 samples)

In [8]:
# Define independent variables (X) and dependent variable (y)
X4 = df4.drop(columns=['Output'])
y4 = df4['Output']

# Add a constant term to the independent variables
X4 = sm.add_constant(X4)
model4 = sm.OLS(y4, X4).fit() # Fit the OLS regression model
print(model4.summary())

                            OLS Regression Results                            
Dep. Variable:                 Output   R-squared:                       0.579
Model:                            OLS   Adj. R-squared:                  0.512
Method:                 Least Squares   F-statistic:                     8.611
Date:                Sun, 09 Aug 2026   Prob (F-statistic):           0.000164
Time:                        18:19:30   Log-Likelihood:                -88.031
No. Observations:                  30   AIC:                             186.1
Df Residuals:                      25   BIC:                             193.1
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -3.0353      2.956     -1.027      0.3

<a id = "5"></a><br>
# Function 5:  A chemical process in a factory (4D array, 20 samples)

In [9]:
# Define independent variables (X) and dependent variable (y)
X5 = df5.drop(columns=['Output'])
y5 = df5['Output']

# Add a constant term to the independent variables
X5 = sm.add_constant(X5)
model5 = sm.OLS(y5, X5).fit() # Fit the OLS regression model
print(model5.summary())

                            OLS Regression Results                            
Dep. Variable:                 Output   R-squared:                       0.662
Model:                            OLS   Adj. R-squared:                  0.572
Method:                 Least Squares   F-statistic:                     7.350
Date:                Sun, 09 Aug 2026   Prob (F-statistic):            0.00174
Time:                        18:19:30   Log-Likelihood:                -127.60
No. Observations:                  20   AIC:                             265.2
Df Residuals:                      15   BIC:                             270.2
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       -348.5110    168.969     -2.063      0.0

<a id = "6"></a><br>
# Function 6:  A cake recipe (5D array, 20 samples)

In [10]:
# Define independent variables (X) and dependent variable (y)
X6 = df6.drop(columns=['Output'])
y6 = df6['Output']

# Add a constant term to the independent variables
X6 = sm.add_constant(X6)
model6 = sm.OLS(y6, X6).fit() # Fit the OLS regression model
print(model6.summary())

                            OLS Regression Results                            
Dep. Variable:                 Output   R-squared:                       0.647
Model:                            OLS   Adj. R-squared:                  0.521
Method:                 Least Squares   F-statistic:                     5.138
Date:                Sun, 09 Aug 2026   Prob (F-statistic):            0.00696
Time:                        18:19:30   Log-Likelihood:                -1.9441
No. Observations:                  20   AIC:                             15.89
Df Residuals:                      14   BIC:                             21.86
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.0954      0.339     -3.231      0.0

<a id = "7"></a><br>
# Function 7:  Unspecified field (6D array, 30 samples)

In [11]:
# Define independent variables (X) and dependent variable (y)
X7 = df7.drop(columns=['Output'])
y7 = df7['Output']

# Add a constant term to the independent variables
X7 = sm.add_constant(X7)
model7 = sm.OLS(y7, X7).fit() # Fit the OLS regression model
print(model7.summary())

                            OLS Regression Results                            
Dep. Variable:                 Output   R-squared:                       0.346
Model:                            OLS   Adj. R-squared:                  0.176
Method:                 Least Squares   F-statistic:                     2.030
Date:                Sun, 09 Aug 2026   Prob (F-statistic):              0.103
Time:                        18:19:30   Log-Likelihood:               -0.28677
No. Observations:                  30   AIC:                             14.57
Df Residuals:                      23   BIC:                             24.38
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.4441      0.238      1.870      0.0

<a id = "8"></a><br>
# Function 8:  Unspecified field (8D array, 40 samples)

In [12]:
# Define independent variables (X) and dependent variable (y)
X8 = df8.drop(columns=['Output'])
y8 = df8['Output']

# Add a constant term to the independent variables
X8 = sm.add_constant(X8)
model8 = sm.OLS(y8, X8).fit() # Fit the OLS regression model
print(model8.summary())


                            OLS Regression Results                            
Dep. Variable:                 Output   R-squared:                       0.899
Model:                            OLS   Adj. R-squared:                  0.873
Method:                 Least Squares   F-statistic:                     34.51
Date:                Sun, 09 Aug 2026   Prob (F-statistic):           2.42e-13
Time:                        18:19:30   Log-Likelihood:                -8.7135
No. Observations:                  40   AIC:                             35.43
Df Residuals:                      31   BIC:                             50.63
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         10.9721      0.314     34.916      0.0